In [2]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import default_rng
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from scipy.constants import G
import scipy.constants as const
import matplotlib as mpl
from scipy.interpolate import interp1d

rng=default_rng()

num_steps=10000

t_span=[0, 100]

t=np.linspace(0, t_span[1], num_steps)

"""The number of bodies is randomly generated. It ranges from 2 to 12. The minimum value of 2 is ensured by the "+2",
   so that there is always some form of interaction."""
n_bodies=int(10*rng.random())+2 

"""The elements of xyz0 are of the form x0_i, y0_i, z0_i, vx0_i, vy0_i, vz0_i, representing the initial coordinates and speeds 
   of the i-th body. """
xyz0=np.zeros(6*n_bodies) 

eps=1e-5 #A very small value that will play a role later.

"""I generate random values for the elements of xyz0. The coordinates and velocities differ in orders of magnitude, 
   so the effect of the forces will be relatively noticeable."""
xyz0[0::6]=((rng.random(n_bodies)-0.5)*10**4).astype(int)
xyz0[1::6]=((rng.random(n_bodies)-0.5)*10**4).astype(int)
xyz0[2::6]=((rng.random(n_bodies)-0.5)*10**4).astype(int)
xyz0[3::6]=((rng.random(n_bodies)-0.5)*10).astype(int)
xyz0[4::6]=((rng.random(n_bodies)-0.5)*10).astype(int)
xyz0[5::6]=((rng.random(n_bodies)-0.5)*10).astype(int)

mass_scale=(rng.random(n_bodies)*10/2).astype(int) #I generate random orders of magnitude for the masses of the bodies.

mass=rng.random(n_bodies)*10**mass_scale+eps #I generate random values for the masses of the bodies. "+eps" ensures that no mass is zero.



In [ ]:
#For the gravitational case, run this cell.

charge=mass #This choice will become clear once we get to the definition of the diff_eq function.
C=-G#The Coulomb's constant

In [ ]:
#For the electrosta case, run this cell.

charge_scale=(rng.random(n_bodies)*10/5).astype(int) #Generating the electrical charges in the same way as the masse. 
charge=(rng.random(n_bodies)-0.5)*10**charge_scale
C = 1.0 / (4.0 * np.pi * const.epsilon_0) # Coulomb's constant

In [ ]:
def diff_eq(t, y, mass, charge, C):

    dy=np.zeros(len(y)) #The elements of dy are the derivatives of y which has the same structure as xyz0.

    #The dericatives of the x, y, z coordinates are vx, vy, vz which are already contained in y. 
    dy[0::6]=y[3::6]
    dy[1::6]=y[4::6]
    dy[2::6]=y[5::6]

    pos=y.reshape(len(mass), 6)[:, :3]

    d_pos=pos[None, :, :] - pos[:, None, :] #Computing x_i-x_j, y_i-y_j and z_i-z_j for each ith and jth body. 
    
    dr_3_initial=np.sqrt(np.sum(d_pos**2, axis=-1))**3 #Computing |r_i-r_j|^3 for each i-th and j-th body.
    
     """Every value smaller than eps is replaced with eps. This is done mostly to avoid division by zero, 
        but also to mitigate numerical energy drift."""
    dr_3=np.maximum(dr_3_initial, eps) 

    """The forces between every i-th and j-th body are computed. For i = j, d_pos is zero, so the forces will be zero as they should be. 
       The forces are proportional to the i-th and j-th charges. For gravitational forces, these charges should be equal to the masses, 
       which explains the previous choice of setting charge = mass"""
    Fij=C*charge[:,None, None]*charge[None, :, None]*d_pos/dr_3[:, :, None] 

    """The acceleration of the i-th body is the sum all of the forces(every F_ij with i constant and j from 1 to n_bodies) acting on it, 
       divided by its mass."""
    a_ij=Fij.sum(axis=1)/mass[:, None]

    dy[3::6]=a_ij[:, 0]

    dy[4::6]=a_ij[:, 1]

    dy[5::6]=a_ij[:, 2]

    return dy


In [ ]:
sol=solve_ivp(fun=diff_eq,
              t_span=t_span,
              y0=xyz0,
              args=[mass, charge, C],
              t_eval=t,
              method='DOP853')

In [ ]:
TARGET_FRAMES = 200

# Original full state array from solve_ivp
states_raw = sol.y.reshape(n_bodies, 6, -1)
pos_raw = states_raw[:, :3, :] # Shape: (n_bodies, 3, original_steps)

# Time vectors for resampling
t_orig = sol.t
t_100 = np.linspace(t_orig[0], t_orig[-1], TARGET_FRAMES)

# Interpolate positions down/up to exactly 100 steps
interp_func = interp1d(t_orig, pos_raw, axis=2, kind="linear")
pos_history = interp_func(t_100) # Shape is GUARANTEED to be (n_bodies, 3, 100)
num_frames = TARGET_FRAMES

pad = 1.0
x_lim = (pos_history[:, 0, :].min() - pad, pos_history[:, 0, :].max() + pad)
y_lim = (pos_history[:, 1, :].min() - pad, pos_history[:, 1, :].max() + pad)
z_lim = (pos_history[:, 2, :].min() - pad, pos_history[:, 2, :].max() + pad)

colors = plt.cm.tab10(np.linspace(0, 1, n_bodies))

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")


def update(frame):
    ax.clear()

    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_zlim(z_lim)

    ax.set_title(
        f"N-Body Simulation (Frame {frame + 1}/{num_frames} | t = {t_100[frame]:.2f}s)"
    )
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    for i in range(n_bodies):
        # Trajectory trail up to current frame
        ax.plot(
            pos_history[i, 0, : frame + 1],
            pos_history[i, 1, : frame + 1],
            pos_history[i, 2, : frame + 1],
            color=colors[i],
            alpha=0.4,
            linestyle="--",
        )
        # Current body position marker
        ax.scatter(
            pos_history[i, 0, frame],
            pos_history[i, 1, frame],
            pos_history[i, 2, frame],
            color=colors[i],
            s=40,
        )


# Always exactly the TARGET_FRAMES frames at 40ms per frame (4 seconds playback)
anim = FuncAnimation(fig, update, frames=TARGET_FRAMES, interval=40)
plt.close(fig)

# Display interactive HTML animation
HTML(anim.to_jshtml())